## Mutation-Level Alternative Statistical Models
This notebook mirrors the main manuscript workflow: shared mutation-level curation first, then Essential/Nonessential/Combined dataset splits, then holdout and temporal reclassification evaluation in one pass.


## 1. Imports and Shared Mutation-Level Setup


In [1]:

import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import accuracy_score, auc, classification_report, confusion_matrix, f1_score, roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

NONESSENTIAL_GENES = {"pncA", "gid", "ethA"}
DATASET_ORDER = ["Essential", "Nonessential", "Combined"]


def resolve_project_root() -> Path:
    candidates = [
        Path.cwd(),
        Path.cwd().parent,
        Path.cwd().parent.parent,
    ]
    for candidate in candidates:
        if (candidate / 'paper_release/source_data/derived_features/2021/2021_final_df.csv').exists():
            return candidate
    raise FileNotFoundError('Could not locate project root containing paper_release/source_data')


PROJECT_ROOT = resolve_project_root()
DATA_2021 = PROJECT_ROOT / 'paper_release/source_data/derived_features/2021/2021_final_df.csv'
DATA_2023 = PROJECT_ROOT / 'paper_release/source_data/derived_features/2023/2023_final_df.csv'
RESULTS_DIR = PROJECT_ROOT / 'Comparison_Model' / 'mutation_level_results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print('Resolved PROJECT_ROOT:', PROJECT_ROOT)

BINARY_MAP = {
    "1) Assoc w R": 1,
    "2) Assoc w R - Interim": 1,
    "4) Not assoc w R - Interim": 0,
    "5) Not assoc w R": 0,
}

CONFIDENCE_PRIORITY = {
    '1) Assoc w R': 0,
    '2) Assoc w R - Interim': 1,
    '4) Not assoc w R - Interim': 2,
    '5) Not assoc w R': 3,
    '3) Uncertain significance': 4,
}


def load_release_features(path):
    df = pd.read_csv(path).copy()
    rename_map = {
        'mutation_oneletter': 'one_letter_mutation',
        'Rosetta_fa_atr': 'fa_atr',
        'Rosetta_fa_rep': 'fa_rep',
        'Rosetta_fa_sol': 'fa_sol',
        'Rosetta_fa_elec': 'fa_elec',
        'Rosetta_fa_dun': 'fa_dun',
        'Rosetta_ddG': 'thermostability',
        'Prox_3D_zeroed': 'Proximity_to_R_Conferring',
        'LLR_score': 'llr_score',
        'freq_variant': 'frequency',
        'AAIndex_mut1': 'mut_AAIndex1',
        'AAIndex_mut2': 'mut_AAIndex2',
        'AAIndex_mut3': 'mut_AAIndex3',
        'AAIndex_mut4': 'mut_AAIndex4',
        'AAIndex_mut5': 'mut_AAIndex5',
        'AAIndex_mut6': 'mut_AAIndex6',
        'AAIndex_mut7': 'mut_AAIndex7',
        'AAIndex_mut8': 'mut_AAIndex8',
    }
    return df.rename(columns=rename_map)


def collapse_supervised_mutation_level(df):
    labeled = df[df['confidence'] != '3) Uncertain significance'].copy()
    labeled['_binary_confidence'] = labeled['confidence'].map(BINARY_MAP)
    labeled = labeled[labeled['_binary_confidence'].notna()].copy()
    pre_n = labeled.shape[0]
    labeled = labeled.drop_duplicates(subset=['gene', 'one_letter_mutation', '_binary_confidence']).copy()
    print(f'Collapsed supervised labeled set: {pre_n} -> {labeled.shape[0]} rows')
    return labeled.drop(columns=['_binary_confidence'])


def collapse_confidence_to_mutation_level(df, confidence_col='confidence'):
    rows = []
    for (gene, mut), sub in df.groupby(['gene', 'one_letter_mutation'], dropna=False):
        confs = [c for c in sub[confidence_col].dropna().tolist()]
        non_uncertain = [c for c in confs if c != '3) Uncertain significance']
        pool = non_uncertain if non_uncertain else confs
        chosen = sorted(pool, key=lambda c: CONFIDENCE_PRIORITY.get(c, 999))[0] if pool else np.nan
        first = sub.iloc[0].copy()
        first[confidence_col] = chosen
        rows.append(first)
    return pd.DataFrame(rows).reset_index(drop=True)


def merge_catalogs_mutation_level(df_2021, df_2023):
    df_2021_mut = collapse_confidence_to_mutation_level(df_2021, 'confidence')
    df_2023_mut = collapse_confidence_to_mutation_level(df_2023, 'confidence')
    merged_df = pd.merge(
        df_2021_mut,
        df_2023_mut,
        on=['gene', 'one_letter_mutation'],
        suffixes=('_2021', '_2023')
    )
    return merged_df.drop_duplicates()


def filter_by_dataset(df, dataset_name, gene_col='gene'):
    if dataset_name == 'Combined':
        return df.copy()
    if dataset_name == 'Nonessential':
        return df[df[gene_col].isin(NONESSENTIAL_GENES)].copy()
    if dataset_name == 'Essential':
        return df[~df[gene_col].isin(NONESSENTIAL_GENES)].copy()
    raise ValueError(f'Unknown dataset: {dataset_name}')


def summarize_binary_metrics(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    sensitivity = tp / (tp + fn) if (tp + fn) else np.nan
    specificity = tn / (tn + fp) if (tn + fp) else np.nan
    return {
        'Support S': int((y_true == 0).sum()),
        'Support R': int((y_true == 1).sum()),
        'Sensitivity': sensitivity,
        'Specificity': specificity,
        'Incorrect': int((y_true != y_pred).sum()),
        'Type I Error': fp / (fp + tn) if (fp + tn) else np.nan,
        'Type II Error': fn / (fn + tp) if (fn + tp) else np.nan,
    }


def build_reclassified_variant_sets(df_2021, df_2023, feature_columns):
    merged_df = merge_catalogs_mutation_level(df_2021, df_2023)
    unchanged_uncertain = merged_df[
        (merged_df['confidence_2021'] == '3) Uncertain significance') &
        (merged_df['confidence_2023'] == '3) Uncertain significance')
    ].drop_duplicates()

    reclassified_from_2021_uncertain = merged_df[
        (merged_df['confidence_2021'] == '3) Uncertain significance') &
        (merged_df['confidence_2023'] != '3) Uncertain significance')
    ].drop_duplicates(subset=['gene', 'one_letter_mutation', 'confidence_2021', 'confidence_2023']).copy()

    reclassified_from_2021_uncertain['confidence_2023'] = reclassified_from_2021_uncertain['confidence_2023'].map(BINARY_MAP)
    reclassified_from_2021_uncertain = reclassified_from_2021_uncertain[
        reclassified_from_2021_uncertain['confidence_2023'].notna()
    ].copy()

    uncertain_2021_df = df_2021[df_2021['confidence'] == '3) Uncertain significance'].copy()
    uncertain_2021_df = uncertain_2021_df.drop_duplicates(subset=['gene', 'one_letter_mutation'])
    reclassified_eval_df = uncertain_2021_df.merge(
        reclassified_from_2021_uncertain[['gene', 'one_letter_mutation', 'confidence_2023']].drop_duplicates(),
        on=['gene', 'one_letter_mutation'],
        how='inner'
    )
    reclassified_eval_df[feature_columns] = reclassified_eval_df[feature_columns].fillna(0)

    return merged_df, unchanged_uncertain, reclassified_from_2021_uncertain, reclassified_eval_df


catalog_data = load_release_features(DATA_2021)
catalog_data = catalog_data.drop_duplicates()
catalog_data = catalog_data[catalog_data['Prox_3D'].notna()].copy()
print('Catalog rows after Prox_3D filter:', catalog_data.shape[0])

full_label_data = collapse_supervised_mutation_level(catalog_data)
full_label_data['binary_confidence'] = full_label_data['confidence'].map(BINARY_MAP)
print('Labeled rows available for modeling:', full_label_data.shape[0])

raw_2021 = load_release_features(DATA_2021)
raw_2023 = load_release_features(DATA_2023)
raw_2021 = raw_2021[raw_2021['Prox_3D'].notna()].copy()
raw_2023 = raw_2023[raw_2023['Prox_3D'].notna()].copy()


Resolved PROJECT_ROOT: /project/pi_annagreen_umass_edu/mahbuba/all_projects/resistance_forecast
Catalog rows after Prox_3D filter: 4248
Collapsed supervised labeled set: 370 -> 345 rows
Labeled rows available for modeling: 345


## 2. Dataset Curation and Gene-Stratified Splits
The dataset split happens here, up front, so the rest of the notebook follows the same three-way flow as the main manuscript notebook.


In [2]:

base_drop_cols = {
    'mutation_wt', 'mutation_pos', 'mutation_mut',
    'drug', 'one_letter_mutation',
    'DeltaZ', 'frequency',
    'Prox_WHO_Adjusted_Pos',
    'Prox_1D', 'Prox_1D_nearest',
    'Prox_3D', 'Prox_3D_nearest',
    'llr_score', 'mutation',
}
base_drop_cols.update({c for c in full_label_data.columns if c.startswith('AAIndex_delta')})
meta_cols = {'confidence', 'phenotype', 'gene', 'binary_confidence'}
allowed_features = {
    'fa_atr', 'fa_rep', 'fa_sol', 'fa_elec', 'fa_dun', 'thermostability',
    'Proximity_to_R_Conferring',
    'mut_AAIndex1', 'mut_AAIndex2', 'mut_AAIndex3', 'mut_AAIndex4',
    'mut_AAIndex5', 'mut_AAIndex6', 'mut_AAIndex7', 'mut_AAIndex8',
    'LLR_dim33', 'LLR_dim46', 'LLR_dim62', 'LLR_dim70', 'LLR_dim124',
    'LLR_dim192', 'LLR_dim207', 'LLR_dim258', 'LLR_dim267', 'LLR_dim315'
}

candidate_numeric_columns = [
    c for c in full_label_data.columns
    if c not in base_drop_cols and c not in meta_cols and pd.api.types.is_numeric_dtype(full_label_data[c])
]
numeric_columns = [c for c in candidate_numeric_columns if c in allowed_features]
print('Feature count:', len(numeric_columns))
print(numeric_columns)


Feature count: 25
['fa_atr', 'fa_rep', 'fa_sol', 'fa_elec', 'fa_dun', 'thermostability', 'Proximity_to_R_Conferring', 'LLR_dim33', 'LLR_dim46', 'LLR_dim62', 'LLR_dim70', 'LLR_dim124', 'LLR_dim192', 'LLR_dim207', 'LLR_dim258', 'LLR_dim267', 'LLR_dim315', 'mut_AAIndex1', 'mut_AAIndex2', 'mut_AAIndex3', 'mut_AAIndex4', 'mut_AAIndex5', 'mut_AAIndex6', 'mut_AAIndex7', 'mut_AAIndex8']


In [3]:

datasets = {}
for dataset_name in DATASET_ORDER:
    labeled_df = filter_by_dataset(full_label_data, dataset_name)
    df_2021_subset = filter_by_dataset(raw_2021, dataset_name)
    df_2023_subset = filter_by_dataset(raw_2023, dataset_name)
    merged_df, unchanged_uncertain, reclassified_from_2021_uncertain, reclassified_eval_df = build_reclassified_variant_sets(
        df_2021_subset, df_2023_subset, numeric_columns
    )
    datasets[dataset_name] = {
        'labeled_df': labeled_df,
        'df_2021': df_2021_subset,
        'df_2023': df_2023_subset,
        'merged_df': merged_df,
        'unchanged_uncertain': unchanged_uncertain,
        'reclassified_from_2021_uncertain': reclassified_from_2021_uncertain,
        'reclassified_eval_df': reclassified_eval_df,
    }

    print('Labeled rows:', labeled_df.shape[0])
    print('Temporal unchanged uncertain:', unchanged_uncertain.shape[0])
    print('Temporal reclassified variants:', reclassified_eval_df.shape[0])
    if not reclassified_eval_df.empty:
        print('Temporal class counts:', reclassified_eval_df['confidence_2023'].value_counts().to_dict())


Labeled rows: 159
Temporal unchanged uncertain: 2294
Temporal reclassified variants: 7
Temporal class counts: {1: 6, 0: 1}
Labeled rows: 186
Temporal unchanged uncertain: 820
Temporal reclassified variants: 55
Temporal class counts: {1: 51, 0: 4}
Labeled rows: 345
Temporal unchanged uncertain: 3114
Temporal reclassified variants: 62
Temporal class counts: {1: 57, 0: 5}


## 3. GAM and MARS Holdout and Temporal Reclassification Evaluation


In [6]:

import scipy.sparse as sp
for cls_name in ['csr_matrix', 'csc_matrix', 'csr_array', 'csc_array']:
    cls = getattr(sp, cls_name, None)
    if cls is not None and not hasattr(cls, 'A'):
        cls.A = property(lambda self: self.toarray())

from pygam import LogisticGAM
import rpy2.robjects as ro
from rpy2.robjects.packages import importr
from rpy2.robjects import pandas2ri

pandas2ri.activate()
earth = importr('earth')
predict_r = ro.r['predict']


def run_gam_and_mars_for_dataset(dataset_name, dataset_bundle):
    labeled_df = dataset_bundle['labeled_df'].copy()
    X_label = labeled_df[numeric_columns].fillna(0)
    y_label = labeled_df['confidence'].map(BINARY_MAP)

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_label)
    X_train, X_test, y_train, y_test = train_test_split(
        X_scaled, y_label, test_size=0.3, random_state=42
    )

    gam_model = LogisticGAM().fit(X_train, y_train)
    gam_holdout_prob = np.asarray(gam_model.predict_proba(X_test)).reshape(-1)
    gam_holdout_pred = (gam_holdout_prob >= 0.5).astype(int)
    gam_holdout_auc = roc_auc_score(y_test, gam_holdout_prob) if len(np.unique(y_test)) == 2 else np.nan
    gam_holdout_metrics = summarize_binary_metrics(np.asarray(y_test), gam_holdout_pred)

    mars_model = earth.earth(X_train, y_train, degree=4)
    mars_holdout_prob = np.asarray(predict_r(mars_model, X_test, type='response')).reshape(-1)
    mars_holdout_pred = np.asarray(predict_r(mars_model, X_test, type='class')).reshape(-1).astype(int)
    mars_holdout_auc = roc_auc_score(y_test, mars_holdout_prob) if len(np.unique(y_test)) == 2 else np.nan
    mars_holdout_metrics = summarize_binary_metrics(np.asarray(y_test), mars_holdout_pred)

    reclassified_eval_df = dataset_bundle['reclassified_eval_df'].copy()
    X_reclassified = scaler.transform(reclassified_eval_df[numeric_columns].fillna(0))
    y_true_temporal = reclassified_eval_df['confidence_2023'].to_numpy()

    gam_temporal_prob = np.asarray(gam_model.predict_proba(X_reclassified)).reshape(-1)
    gam_temporal_pred = (gam_temporal_prob >= 0.5).astype(int)
    gam_temporal_auc = roc_auc_score(y_true_temporal, gam_temporal_prob) if len(np.unique(y_true_temporal)) == 2 else np.nan
    gam_temporal_metrics = summarize_binary_metrics(y_true_temporal, gam_temporal_pred)

    mars_temporal_prob = np.asarray(predict_r(mars_model, X_reclassified, type='response')).reshape(-1)
    mars_temporal_pred = np.asarray(predict_r(mars_model, X_reclassified, type='class')).reshape(-1).astype(int)
    mars_temporal_auc = roc_auc_score(y_true_temporal, mars_temporal_prob) if len(np.unique(y_true_temporal)) == 2 else np.nan
    mars_temporal_metrics = summarize_binary_metrics(y_true_temporal, mars_temporal_pred)

    common = {
        'Dataset': dataset_name,
        'Labeled Samples': int(labeled_df.shape[0]),
        'Reclassified Variants': int(reclassified_eval_df.shape[0]),
    }
    return [
        {
            **common,
            'Model': 'GAM',
            'Holdout AUC': gam_holdout_auc,
            'Holdout Accuracy': accuracy_score(y_test, gam_holdout_pred),
            'Holdout Sens (R)': gam_holdout_metrics['Sensitivity'],
            'Holdout Spec (S)': gam_holdout_metrics['Specificity'],
            'Holdout Support R': gam_holdout_metrics['Support R'],
            'Holdout Support S': gam_holdout_metrics['Support S'],
            'Temporal AUC': gam_temporal_auc,
            'Temporal Accuracy': accuracy_score(y_true_temporal, gam_temporal_pred),
            'Temporal F1 (R)': f1_score(y_true_temporal, gam_temporal_pred, pos_label=1, zero_division=0),
            'Temporal Sens (R)': gam_temporal_metrics['Sensitivity'],
            'Temporal Spec (S)': gam_temporal_metrics['Specificity'],
            'Temporal Support R': gam_temporal_metrics['Support R'],
            'Temporal Support S': gam_temporal_metrics['Support S'],
            'Temporal Incorrect': gam_temporal_metrics['Incorrect'],
            'Type I Error': gam_temporal_metrics['Type I Error'],
            'Type II Error': gam_temporal_metrics['Type II Error'],
        },
        {
            **common,
            'Model': 'MARS',
            'Holdout AUC': mars_holdout_auc,
            'Holdout Accuracy': accuracy_score(y_test, mars_holdout_pred),
            'Holdout Sens (R)': mars_holdout_metrics['Sensitivity'],
            'Holdout Spec (S)': mars_holdout_metrics['Specificity'],
            'Holdout Support R': mars_holdout_metrics['Support R'],
            'Holdout Support S': mars_holdout_metrics['Support S'],
            'Temporal AUC': mars_temporal_auc,
            'Temporal Accuracy': accuracy_score(y_true_temporal, mars_temporal_pred),
            'Temporal F1 (R)': f1_score(y_true_temporal, mars_temporal_pred, pos_label=1, zero_division=0),
            'Temporal Sens (R)': mars_temporal_metrics['Sensitivity'],
            'Temporal Spec (S)': mars_temporal_metrics['Specificity'],
            'Temporal Support R': mars_temporal_metrics['Support R'],
            'Temporal Support S': mars_temporal_metrics['Support S'],
            'Temporal Incorrect': mars_temporal_metrics['Incorrect'],
            'Type I Error': mars_temporal_metrics['Type I Error'],
            'Type II Error': mars_temporal_metrics['Type II Error'],
        },
    ]


In [7]:

comparison_summary_df = pd.DataFrame([
    row
    for dataset_name in DATASET_ORDER
    for row in run_gam_and_mars_for_dataset(dataset_name, datasets[dataset_name])
])

comparison_summary_df.to_csv(RESULTS_DIR / 'gam_mars_mutation_level_summary.csv', index=False)
comparison_summary_df


did not converge


/work/pi_annagreen_umass_edu/mahbuba/esmfold/lib/python3.10/site-packages/pygam/links.py:149: RuntimeWarning: divide by zero encountered in divide
  return dist.levels / (mu * (dist.levels - mu))
/work/pi_annagreen_umass_edu/mahbuba/esmfold/lib/python3.10/site-packages/pygam/pygam.py:627: RuntimeWarning: invalid value encountered in multiply
  self.link.gradient(mu, self.distribution) ** 2


,Dataset,Labeled Samples,Reclassified Variants,Model,Holdout AUC,Holdout Accuracy,Holdout Sens (R),Holdout Spec (S),Holdout Support R,Holdout Support S,Temporal AUC,Temporal Accuracy,Temporal F1 (R),Temporal Sens (R),Temporal Spec (S),Temporal Support R,Temporal Support S,Temporal Incorrect,Type I Error,Type II Error
0,Essential,159,7,GAM,0.709402,0.812500,1.000000,0.000000,39,9,1.000000,0.857143,0.923077,1.000000,0.0,6,1,1,1.0,0.000000
1,Essential,159,7,MARS,0.917379,0.958333,0.974359,0.888889,39,9,0.833333,0.857143,0.909091,0.833333,1.0,6,1,1,0.0,0.166667
2,Nonessential,186,55,GAM,0.443149,0.875000,1.000000,0.000000,49,7,0.465686,0.927273,0.962264,1.000000,0.0,51,4,4,1.0,0.000000
3,Nonessential,186,55,MARS,0.583090,0.821429,0.938776,0.000000,49,7,0.274510,0.872727,0.932039,0.941176,0.0,51,4,7,1.0,0.058824
4,Combined,345,62,GAM,0.862745,0.855769,1.000000,0.117647,87,17,0.557895,0.935484,0.966102,1.000000,0.2,57,5,4,0.8,0.000000
5,Combined,345,62,MARS,0.879310,0.903846,0.965517,0.588235,87,17,0.821053,0.870968,0.928571,0.912281,0.4,57,5,8,0.6,0.087719
